# Laboratorio 5: Modelos de lenguaje

En este laboratorio se construyen modelos de lenguaje basados en n-gramas usando el texto de *Don Quijote de la Mancha*.  
El objetivo es preparar el corpus, construir modelos unigrama, bigrama y trigrama, aplicar suavizado, evaluar con perplejidad y probar una función simple de autocompletado.

## Importación de librerías

Primero se importan las librerías necesarias para leer el texto, limpiar el corpus, contar palabras y separar los datos en entrenamiento, validación y prueba.

In [3]:
# Librerías principales para manejo de texto, conteos y división de datos
import re
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

## Carga del corpus

Se carga el texto completo de *Don Quijote de la Mancha*.  
Antes de procesarlo, se muestra una pequeña parte para verificar que el archivo se leyó correctamente.

In [1]:
# Cargamos el texto completo de Don Quijote
with open("don-quijote.txt", "r", encoding="utf-8") as file:
    texto = file.read()

# Mostramos una parte pequeña del texto para verificar que se cargó bien
print(texto[:1000])

The Project Gutenberg EBook of Don Quijote, by Miguel de Cervantes Saavedra

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: Don Quijote

Author: Miguel de Cervantes Saavedra

Posting Date: April 27, 2010 [EBook #2000]
Release Date: December, 1999

Language: Spanish


*** START OF THIS PROJECT GUTENBERG EBOOK DON QUIJOTE ***




Produced by an anonymous Project Gutenberg volunteer. Text
file corrections and new HTML file by Joaquin Cuenca Abela.











El ingenioso hidalgo don Quijote de la Mancha


TASA

Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de
los que residen en su Consejo, certifico y doy fe que, habiendo visto por
los señores dél un libro intitulado El ingenioso hidalgo de la Mancha,
compuesto por Miguel de Cervantes Saavedra, tasaron cada


## Limpieza básica

En esta parte solo se normalizan espacios y saltos de línea.  
No se eliminan stopwords ni se aplica lematización, porque el objetivo es conservar el orden natural de las palabras para construir modelos de lenguaje.

In [4]:
# Reemplazamos saltos de línea múltiples por espacios
texto_limpio = re.sub(r"\s+", " ", texto)

# Quitamos espacios al inicio y al final
texto_limpio = texto_limpio.strip()

# Verificamos el resultado
print(texto_limpio[:1000])

The Project Gutenberg EBook of Don Quijote, by Miguel de Cervantes Saavedra This eBook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.net Title: Don Quijote Author: Miguel de Cervantes Saavedra Posting Date: April 27, 2010 [EBook #2000] Release Date: December, 1999 Language: Spanish *** START OF THIS PROJECT GUTENBERG EBOOK DON QUIJOTE *** Produced by an anonymous Project Gutenberg volunteer. Text file corrections and new HTML file by Joaquin Cuenca Abela. El ingenioso hidalgo don Quijote de la Mancha TASA Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de los que residen en su Consejo, certifico y doy fe que, habiendo visto por los señores dél un libro intitulado El ingenioso hidalgo de la Mancha, compuesto por Miguel de Cervantes Saavedra, tasaron cada pliego del dicho libro a t


## Segmentación en oraciones

El texto se divide en oraciones porque los modelos de lenguaje trabajan con secuencias.  
Cada oración será tratada como una secuencia independiente de palabras.

In [6]:
# Dividimos el texto en oraciones usando puntos, signos de pregunta y exclamación
oraciones = re.split(r'(?<=[.!?¿¡])\s+', texto_limpio)

# Eliminamos oraciones vacías
oraciones = [oracion.strip() for oracion in oraciones if oracion.strip()]

# Mostramos cuántas oraciones se obtuvieron
print("Cantidad de oraciones:", len(oraciones))

# Ejemplo de algunas oraciones
for i in range(5):
    print(f"{i+1}.", oraciones[i])

Cantidad de oraciones: 9577
1. ﻿The Project Gutenberg EBook of Don Quijote, by Miguel de Cervantes Saavedra This eBook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever.
2. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.net Title: Don Quijote Author: Miguel de Cervantes Saavedra Posting Date: April 27, 2010 [EBook #2000] Release Date: December, 1999 Language: Spanish *** START OF THIS PROJECT GUTENBERG EBOOK DON QUIJOTE *** Produced by an anonymous Project Gutenberg volunteer.
3. Text file corrections and new HTML file by Joaquin Cuenca Abela.
4. El ingenioso hidalgo don Quijote de la Mancha TASA Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de los que residen en su Consejo, certifico y doy fe que, habiendo visto por los señores dél un libro intitulado El ingenioso hidalgo de la Mancha, compuesto por Miguel de Cervantes Saavedra,

## Tokenización

Cada oración se convierte en una lista de tokens.  
En este caso, se usan minúsculas para reducir el tamaño del vocabulario, pero no se eliminan palabras ni se cambia su forma original mediante lematización.

In [7]:
# Función para tokenizar una oración en palabras
def tokenizar(oracion):
    # Convertimos a minúsculas para reducir variaciones del vocabulario
    oracion = oracion.lower()
    
    # Extraemos palabras y algunos signos de puntuación como tokens
    tokens = re.findall(r'\w+|[^\w\s]', oracion, re.UNICODE)
    
    return tokens

In [8]:
# Probamos la tokenización con una oración de ejemplo
ejemplo = oraciones[0]
tokens_ejemplo = tokenizar(ejemplo)

print("Oración original:")
print(ejemplo)

print("\nTokens:")
print(tokens_ejemplo)

Oración original:
﻿The Project Gutenberg EBook of Don Quijote, by Miguel de Cervantes Saavedra This eBook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever.

Tokens:
['\ufeff', 'the', 'project', 'gutenberg', 'ebook', 'of', 'don', 'quijote', ',', 'by', 'miguel', 'de', 'cervantes', 'saavedra', 'this', 'ebook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever', '.']


## Tokens de inicio y fin

A cada oración se le agrega el token `<s>` al inicio y el token `</s>` al final.  
Esto permite que el modelo aprenda cómo suelen iniciar y terminar las oraciones.

In [9]:
# Tokenizamos todas las oraciones y agregamos tokens de inicio y fin
oraciones_tokenizadas = []

for oracion in oraciones:
    tokens = tokenizar(oracion)
    
    if len(tokens) > 0:
        tokens_con_marcas = ["<s>"] + tokens + ["</s>"]
        oraciones_tokenizadas.append(tokens_con_marcas)

# Mostramos un ejemplo
print(oraciones_tokenizadas[0])

['<s>', '\ufeff', 'the', 'project', 'gutenberg', 'ebook', 'of', 'don', 'quijote', ',', 'by', 'miguel', 'de', 'cervantes', 'saavedra', 'this', 'ebook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever', '.', '</s>']


## División del corpus

El corpus se divide aleatoriamente en tres conjuntos: entrenamiento, validación y prueba.  
El conjunto de entrenamiento se usa para construir los modelos, validación para comparar resultados y prueba para evaluar el mejor modelo al final.

In [10]:
# Fijamos una semilla para que los resultados sean reproducibles
random.seed(42)

# Primero separamos entrenamiento y un conjunto temporal
train_sentences, temp_sentences = train_test_split(
    oraciones_tokenizadas,
    test_size=0.20,
    random_state=42
)

# Luego dividimos el temporal en validación y prueba
val_sentences, test_sentences = train_test_split(
    temp_sentences,
    test_size=0.50,
    random_state=42
)

print("Oraciones de entrenamiento:", len(train_sentences))
print("Oraciones de validación:", len(val_sentences))
print("Oraciones de prueba:", len(test_sentences))

Oraciones de entrenamiento: 7661
Oraciones de validación: 958
Oraciones de prueba: 958


## Vocabulario de entrenamiento

El vocabulario se construye usando únicamente el conjunto de entrenamiento.  
Esto permite medir después cuántas palabras del conjunto de prueba no fueron vistas durante el entrenamiento.

In [11]:
# Unimos todos los tokens del conjunto de entrenamiento
tokens_train = []

for oracion in train_sentences:
    tokens_train.extend(oracion)

# Creamos el vocabulario como conjunto de palabras únicas
vocab_train = set(tokens_train)

print("Tamaño del vocabulario de entrenamiento:", len(vocab_train))

Tamaño del vocabulario de entrenamiento: 21150


## Palabras no vistas en prueba

Se calcula cuántas palabras del conjunto de prueba no aparecen en el vocabulario del conjunto de entrenamiento.  
Estas palabras se conocen como OOV, que significa palabras fuera del vocabulario.

In [12]:
# Unimos todos los tokens del conjunto de prueba
tokens_test = []

for oracion in test_sentences:
    tokens_test.extend(oracion)

# Contamos cuántos tokens de prueba no están en el vocabulario de entrenamiento
oov_tokens = [token for token in tokens_test if token not in vocab_train]

# Calculamos la proporción de palabras no vistas
proporcion_oov = len(oov_tokens) / len(tokens_test)

print("Total de tokens en prueba:", len(tokens_test))
print("Tokens OOV:", len(oov_tokens))
print("Proporción OOV:", proporcion_oov)
print("Proporción OOV (%):", proporcion_oov * 100)

Total de tokens en prueba: 46397
Tokens OOV: 1363
Proporción OOV: 0.029376899368493654
Proporción OOV (%): 2.9376899368493654


### Relación entre palabras no vistas y data sparsity

Las palabras nunca vistas son aquellas que aparecen en el conjunto de prueba, pero no aparecieron en el conjunto de entrenamiento. Esto representa un problema porque el modelo no tiene información previa para calcular una probabilidad confiable para esas palabras.

Este problema se relaciona con la dispersión de datos, porque en lenguaje natural existen muchísimas palabras y combinaciones posibles, pero muchas aparecen muy pocas veces o no aparecen en el corpus de entrenamiento. Entonces, aunque el texto sea grande, no siempre contiene todos los casos posibles.

En los modelos de lenguaje basados en conteos, esto puede causar que algunas palabras o secuencias tengan conteo cero. Como consecuencia, el modelo puede asignar probabilidad cero a una oración, aunque esa oración sí sea válida en el idioma.

## 2. Construcción de modelos n-grama

En esta sección se construyen tres modelos de lenguaje basados en conteos: unigrama, bigrama y trigrama.

Un modelo unigrama calcula la probabilidad de una palabra sin tomar en cuenta el contexto.  
Un modelo bigrama calcula la probabilidad de una palabra usando la palabra anterior.  
Un modelo trigrama calcula la probabilidad de una palabra usando las dos palabras anteriores.

Estos modelos se construyen usando únicamente el conjunto de entrenamiento.

### Conteo de n-gramas

Primero se cuentan los unigramas, bigramas y trigramas del conjunto de entrenamiento.  
Estos conteos serán la base para calcular las probabilidades de cada modelo.

In [13]:
# Diccionarios para guardar los conteos de cada tipo de n-grama
conteo_unigramas = Counter()
conteo_bigramas = Counter()
conteo_trigramas = Counter()

# Recorremos cada oración del conjunto de entrenamiento
for oracion in train_sentences:
    
    # Conteo de unigramas
    for token in oracion:
        conteo_unigramas[token] += 1
    
    # Conteo de bigramas
    for i in range(len(oracion) - 1):
        bigrama = (oracion[i], oracion[i + 1])
        conteo_bigramas[bigrama] += 1
    
    # Conteo de trigramas
    for i in range(len(oracion) - 2):
        trigrama = (oracion[i], oracion[i + 1], oracion[i + 2])
        conteo_trigramas[trigrama] += 1

# Mostramos algunos resultados
print("Cantidad de unigramas únicos:", len(conteo_unigramas))
print("Cantidad de bigramas únicos:", len(conteo_bigramas))
print("Cantidad de trigramas únicos:", len(conteo_trigramas))

Cantidad de unigramas únicos: 21150
Cantidad de bigramas únicos: 126415
Cantidad de trigramas únicos: 251610


### Ejemplos de conteos

Se muestran algunos de los n-gramas más frecuentes para revisar qué palabras y secuencias aparecen más en el corpus.

In [14]:
# Mostramos los 10 unigramas más frecuentes
print("Unigramas más frecuentes:")
print(conteo_unigramas.most_common(10))

# Mostramos los 10 bigramas más frecuentes
print("\nBigramas más frecuentes:")
print(conteo_bigramas.most_common(10))

# Mostramos los 10 trigramas más frecuentes
print("\nTrigramas más frecuentes:")
print(conteo_trigramas.most_common(10))

Unigramas más frecuentes:
[(',', 32377), ('que', 16577), ('de', 14612), ('y', 14545), ('la', 8307), ('a', 7986), ('<s>', 7661), ('</s>', 7661), ('.', 6716), ('en', 6548)]

Bigramas más frecuentes:
[(('.', '</s>'), 6489), ((',', 'y'), 4955), ((',', 'que'), 3417), (('<s>', '-'), 1830), (('don', 'quijote'), 1750), (('de', 'la'), 1663), ((';', 'y'), 1581), (('y', ','), 1271), (('lo', 'que'), 1244), (('que', 'no'), 1039)]

Trigramas más frecuentes:
[((',', 'y', ','), 524), ((';', 'y', ','), 472), (('don', 'quijote', '-'), 435), (('don', 'quijote', ','), 432), ((',', 'y', 'que'), 378), ((',', 'que', 'no'), 360), (('dijo', ':', '-'), 346), ((',', 'y', 'no'), 278), (('y', 'así', ','), 277), (('<s>', 'y', ','), 251)]


### Modelo unigrama

El modelo unigrama calcula la probabilidad de una palabra usando su frecuencia relativa en el corpus de entrenamiento.  
Este modelo no toma en cuenta las palabras anteriores, por lo que ignora el contexto.

In [15]:
# Total de tokens en el conjunto de entrenamiento
total_tokens_train = sum(conteo_unigramas.values())

# Función para calcular la probabilidad unigrama
def probabilidad_unigrama(palabra):
    return conteo_unigramas[palabra] / total_tokens_train

In [16]:
# Ejemplo de probabilidad unigrama
palabra_ejemplo = "don"

print("Palabra:", palabra_ejemplo)
print("Probabilidad unigrama:", probabilidad_unigrama(palabra_ejemplo))

Palabra: don
Probabilidad unigrama: 0.005643388681443818


### Modelo bigrama

El modelo bigrama estima la probabilidad de una palabra usando solamente la palabra anterior.  
Esto permite agregar un poco de contexto, pero sigue siendo un modelo simple.

In [17]:
# Función para calcular la probabilidad bigrama
def probabilidad_bigrama(palabra_anterior, palabra_actual):
    bigrama = (palabra_anterior, palabra_actual)
    
    # Si la palabra anterior no existe en entrenamiento, no podemos estimar la probabilidad
    if conteo_unigramas[palabra_anterior] == 0:
        return 0
    
    return conteo_bigramas[bigrama] / conteo_unigramas[palabra_anterior]

In [18]:
# Ejemplo de probabilidad bigrama
palabra_anterior = "don"
palabra_actual = "quijote"

print("Bigrama:", (palabra_anterior, palabra_actual))
print("Probabilidad bigrama:", probabilidad_bigrama(palabra_anterior, palabra_actual))

Bigrama: ('don', 'quijote')
Probabilidad bigrama: 0.8212106992022524


### Modelo trigrama

El modelo trigrama estima la probabilidad de una palabra usando las dos palabras anteriores como contexto.  
Este modelo puede capturar más información que el bigrama, pero también puede tener más problemas cuando una combinación no apareció en entrenamiento.

In [19]:
# Función para calcular la probabilidad trigrama
def probabilidad_trigrama(palabra_1, palabra_2, palabra_actual):
    trigrama = (palabra_1, palabra_2, palabra_actual)
    contexto = (palabra_1, palabra_2)
    
    # Si el contexto de dos palabras no existe, no podemos estimar la probabilidad
    if conteo_bigramas[contexto] == 0:
        return 0
    
    return conteo_trigramas[trigrama] / conteo_bigramas[contexto]

In [23]:
# Ejemplo de probabilidad trigrama
palabra_1 = "don"
palabra_2 = "quijote"
palabra_actual = "dijo"

print("Trigrama:", (palabra_1, palabra_2, palabra_actual))
print("Probabilidad trigrama:", probabilidad_trigrama(palabra_1, palabra_2, palabra_actual))

Trigrama: ('don', 'quijote', 'dijo')
Probabilidad trigrama: 0.0034285714285714284


### Oración de ejemplo

Se selecciona una oración del conjunto de validación para calcular su probabilidad completa usando los tres modelos.

In [33]:
# Buscamos oraciones cortas del conjunto de validación
oraciones_cortas_val = []

for oracion in val_sentences:
    if 5 <= len(oracion) <= 12:
        oraciones_cortas_val.append(oracion)

# Mostramos algunas opciones para escoger una oración más clara
for i, oracion in enumerate(oraciones_cortas_val[:10]):
    print(i, oracion)

0 ['<s>', '-', 'así', 'es', '-', 'dijo', 'sancho', '.', '</s>']
1 ['<s>', '¡', 'bonita', 'es', 'la', 'niña', '!', '</s>']
2 ['<s>', 'capítulo', 'xxvi', '.', '</s>']
3 ['<s>', '-', '¡', 'hola', ',', 'hermano', 'correo', '!', '</s>']
4 ['<s>', '-', '¡', 'a', 'mi', 'mujer', 'con', 'eso', '!', '</s>']
5 ['<s>', '-', '¡', 'milagro', '!', '</s>']
6 ['<s>', '¿', 'cómo', 'y', 'no', 'consideráis', 'que', 'está', 'electo', 'gobernador', '?', '</s>']
7 ['<s>', 'capítulo', 'lxxi', '.', '</s>']
8 ['<s>', 'capítulo', 'xlviii', '.', '</s>']
9 ['<s>', '-', '¡', 'ya', 'quisiera', 'yo', 'ver', 'eso', '!', '</s>']


In [34]:
# Buscamos oraciones del conjunto de validación que mencionen a Don Quijote
oraciones_don_quijote = []

for oracion in val_sentences:
    if "don" in oracion or "quijote" in oracion:
        if 5 <= len(oracion) <= 20:
            oraciones_don_quijote.append(oracion)

# Mostramos algunas opciones
for i, oracion in enumerate(oraciones_don_quijote[:10]):
    print(i, oracion)

0 ['<s>', 'pero', 'don', 'fernando', ',', 'cardenio', 'y', 'el', 'cura', 'le', 'hicieron', 'más', 'llanos', 'y', 'más', 'cortesanos', 'ofrecimientos', '.', '</s>']
1 ['<s>', '-', 'de', 'los', 'afligidos', '-', 'respondió', 'don', 'quijote', '.', '</s>']
2 ['<s>', '-', 'preguntó', 'don', 'quijote', '.', '</s>']
3 ['<s>', 'volvió', 'la', 'hoja', 'don', 'quijote', 'y', 'dijo', ':', '-', 'esto', 'es', 'prosa', ',', 'y', 'parece', 'carta', '.', '</s>']
4 ['<s>', '-', 'socarrón', 'sois', ',', 'sancho', '-', 'respondió', 'don', 'quijote', '-', '.', '</s>']
5 ['<s>', '-', 'confieso', '-', 'dijo', 'don', 'quijote', '-', 'que', 'todo', 'lo', 'que', 'dices', ',', 'sancho', ',', 'sea', 'verdad', '.', '</s>']
6 ['<s>', '-', 'dijo', 'don', 'quijote', '-', '.', '</s>']
7 ['<s>', '-', 'dijo', 'don', 'quijote', '-', '.', '</s>']
8 ['<s>', '-', 'calla', '-', 'dijo', 'don', 'quijote', '-', '.', '</s>']


In [35]:
# Seleccionamos una oración relacionada con Don Quijote
oracion_ejemplo = oraciones_don_quijote[2]

print("Oración de ejemplo:")
print(oracion_ejemplo)

Oración de ejemplo:
['<s>', '-', 'preguntó', 'don', 'quijote', '.', '</s>']


In [36]:
def probabilidad_oracion_unigrama(oracion):
    probabilidad = 1
    
    # Multiplicamos la probabilidad de cada palabra
    for palabra in oracion:
        probabilidad *= probabilidad_unigrama(palabra)
    
    return probabilidad

In [37]:
prob_uni = probabilidad_oracion_unigrama(oracion_ejemplo)

print("Probabilidad de la oración con unigrama:")
print(prob_uni)

Probabilidad de la oración con unigrama:
8.862614682908665e-16


In [38]:
def probabilidad_oracion_bigrama(oracion):
    probabilidad = 1
    
    # Multiplicamos la probabilidad de cada palabra dada la palabra anterior
    for i in range(1, len(oracion)):
        palabra_anterior = oracion[i - 1]
        palabra_actual = oracion[i]
        
        probabilidad *= probabilidad_bigrama(palabra_anterior, palabra_actual)
    
    return probabilidad

In [39]:
prob_bi = probabilidad_oracion_bigrama(oracion_ejemplo)

print("Probabilidad de la oración con bigrama:")
print(prob_bi)

Probabilidad de la oración con bigrama:
1.8573517460043862e-05


In [40]:
def probabilidad_oracion_trigrama(oracion):
    probabilidad = 1
    
    # Multiplicamos la probabilidad de cada palabra usando las dos palabras anteriores
    for i in range(2, len(oracion)):
        palabra_1 = oracion[i - 2]
        palabra_2 = oracion[i - 1]
        palabra_actual = oracion[i]
        
        probabilidad *= probabilidad_trigrama(palabra_1, palabra_2, palabra_actual)
    
    return probabilidad

In [41]:
prob_tri = probabilidad_oracion_trigrama(oracion_ejemplo)

print("Probabilidad de la oración con trigrama:")
print(prob_tri)

Probabilidad de la oración con trigrama:
0.00044837130083031723


### Comparación de probabilidades

En la tabla se comparan las probabilidades asignadas por cada modelo a la misma oración.  
Es posible que algunos modelos asignen probabilidad cero si encuentran una palabra o combinación que no apareció en entrenamiento.

In [42]:
# Creamos una tabla simple con las probabilidades obtenidas
resultados_probabilidades = pd.DataFrame({
    "Modelo": ["Unigrama", "Bigrama", "Trigrama"],
    "Probabilidad": [prob_uni, prob_bi, prob_tri]
})

resultados_probabilidades

,Modelo,Probabilidad
0,Unigrama,8.862615e-16
1,Bigrama,1.857352e-05
2,Trigrama,4.483713e-04


### Análisis de los resultados

En la tabla se observa que el modelo trigrama obtuvo la probabilidad más alta para la oración seleccionada.  
Esto puede pasar porque la oración contiene una secuencia común del texto, como “preguntó don quijote”.

El modelo bigrama también usa contexto, pero solo toma en cuenta la palabra anterior.  
En cambio, el modelo unigrama no usa contexto, por eso su probabilidad fue mucho más baja.

Estos resultados muestran que usar más contexto puede ayudar al modelo, aunque también puede causar problemas cuando aparecen combinaciones que no fueron vistas en entrenamiento.

### Supuesto de Markov

La regla de la cadena permite calcular la probabilidad de una oración usando todas las palabras anteriores como contexto.  
El problema es que esto es difícil de estimar con datos reales, porque casi nunca se repite exactamente el mismo historial largo de palabras en el corpus.

El supuesto de Markov simplifica este problema al usar solo una parte corta del contexto anterior.  
Por ejemplo, el modelo bigrama usa solo la palabra anterior y el modelo trigrama usa las dos palabras anteriores.

Esto hace que el cálculo ya no sea exacto, sino una aproximación.  
Sin embargo, es una aproximación útil porque los contextos cortos sí aparecen con más frecuencia en los datos y se pueden estimar mediante conteos.

## 3. Suavizado

En esta sección se implementa suavizado para el modelo de bigramas.  
El suavizado ayuda a evitar que una oración tenga probabilidad cero cuando aparece una combinación de palabras que no fue vista en entrenamiento.

### Bigrama sin suavizado

Primero se define la probabilidad del modelo bigrama sin suavizado.  
En este caso, si un bigrama no apareció en entrenamiento, su probabilidad será cero.

In [43]:
# Probabilidad bigrama sin suavizado
def probabilidad_bigrama_sin_suavizado(palabra_anterior, palabra_actual):
    bigrama = (palabra_anterior, palabra_actual)
    
    # Si la palabra anterior no aparece, la probabilidad es cero
    if conteo_unigramas[palabra_anterior] == 0:
        return 0
    
    return conteo_bigramas[bigrama] / conteo_unigramas[palabra_anterior]

### Suavizado de Laplace

El suavizado de Laplace suma 1 a todos los conteos de bigramas.  
Esto evita que un bigrama no visto tenga probabilidad cero.

In [44]:
# Tamaño del vocabulario de entrenamiento
V = len(vocab_train)

# Probabilidad bigrama con suavizado de Laplace
def probabilidad_bigrama_laplace(palabra_anterior, palabra_actual):
    bigrama = (palabra_anterior, palabra_actual)
    
    numerador = conteo_bigramas[bigrama] + 1
    denominador = conteo_unigramas[palabra_anterior] + V
    
    return numerador / denominador

### Suavizado add-k

El suavizado add-k funciona parecido a Laplace, pero permite usar valores menores que 1.  
Esto hace que el modelo dé probabilidad a combinaciones no vistas, pero sin modificar tanto las probabilidades originales.

In [45]:
# Probabilidad bigrama con suavizado add-k
def probabilidad_bigrama_add_k(palabra_anterior, palabra_actual, k):
    bigrama = (palabra_anterior, palabra_actual)
    
    numerador = conteo_bigramas[bigrama] + k
    denominador = conteo_unigramas[palabra_anterior] + (k * V)
    
    return numerador / denominador

In [46]:
# Bigrama que probablemente sí aparece en el corpus
bigrama_visto = ("don", "quijote")

# Bigrama inventado o poco probable
bigrama_no_visto = ("quijote", "computadora")

print("Bigrama visto:", bigrama_visto)
print("Sin suavizado:", probabilidad_bigrama_sin_suavizado(bigrama_visto[0], bigrama_visto[1]))
print("Laplace:", probabilidad_bigrama_laplace(bigrama_visto[0], bigrama_visto[1]))
print("Add-k 0.5:", probabilidad_bigrama_add_k(bigrama_visto[0], bigrama_visto[1], 0.5))
print("Add-k 0.1:", probabilidad_bigrama_add_k(bigrama_visto[0], bigrama_visto[1], 0.1))

print("\nBigrama no visto:", bigrama_no_visto)
print("Sin suavizado:", probabilidad_bigrama_sin_suavizado(bigrama_no_visto[0], bigrama_no_visto[1]))
print("Laplace:", probabilidad_bigrama_laplace(bigrama_no_visto[0], bigrama_no_visto[1]))
print("Add-k 0.5:", probabilidad_bigrama_add_k(bigrama_no_visto[0], bigrama_no_visto[1], 0.5))
print("Add-k 0.1:", probabilidad_bigrama_add_k(bigrama_no_visto[0], bigrama_no_visto[1], 0.1))

Bigrama visto: ('don', 'quijote')
Sin suavizado: 0.8212106992022524
Laplace: 0.07521154589579486
Add-k 0.5: 0.13776955768928065
Add-k 0.1: 0.4121761658031088

Bigrama no visto: ('quijote', 'computadora')
Sin suavizado: 0.0
Laplace: 4.3656683838295645e-05
Add-k 0.5: 4.054821182385857e-05
Add-k 0.1: 2.5833118057349522e-05


### Análisis de bigramas vistos y no vistos

En el bigrama visto `("don", "quijote")`, la probabilidad sin suavizado es alta porque esa combinación aparece muchas veces en el corpus.  
Cuando se aplica suavizado, la probabilidad baja un poco porque el modelo reparte parte de la probabilidad hacia combinaciones que no fueron vistas.

En el bigrama no visto `("quijote", "computadora")`, la probabilidad sin suavizado es cero porque esa combinación no aparece en entrenamiento.  
Con Laplace y add-k, la probabilidad ya no es cero, sino un valor pequeño.

Esto muestra que el suavizado permite que el modelo pueda evaluar combinaciones nuevas sin decir que son imposibles.

In [47]:
# Probabilidad de una oración usando bigramas sin suavizado
def probabilidad_oracion_bigrama_sin_suavizado(oracion):
    probabilidad = 1
    
    for i in range(1, len(oracion)):
        palabra_anterior = oracion[i - 1]
        palabra_actual = oracion[i]
        
        probabilidad *= probabilidad_bigrama_sin_suavizado(palabra_anterior, palabra_actual)
    
    return probabilidad

In [48]:
# Probabilidad de una oración usando bigramas con suavizado de Laplace
def probabilidad_oracion_bigrama_laplace(oracion):
    probabilidad = 1
    
    for i in range(1, len(oracion)):
        palabra_anterior = oracion[i - 1]
        palabra_actual = oracion[i]
        
        probabilidad *= probabilidad_bigrama_laplace(palabra_anterior, palabra_actual)
    
    return probabilidad

In [49]:
# Probabilidad de una oración usando bigramas con suavizado add-k
def probabilidad_oracion_bigrama_add_k(oracion, k):
    probabilidad = 1
    
    for i in range(1, len(oracion)):
        palabra_anterior = oracion[i - 1]
        palabra_actual = oracion[i]
        
        probabilidad *= probabilidad_bigrama_add_k(palabra_anterior, palabra_actual, k)
    
    return probabilidad

In [50]:
# Calculamos la probabilidad de la misma oración con distintos tipos de suavizado
prob_sin_suavizado = probabilidad_oracion_bigrama_sin_suavizado(oracion_ejemplo)
prob_laplace = probabilidad_oracion_bigrama_laplace(oracion_ejemplo)
prob_add_05 = probabilidad_oracion_bigrama_add_k(oracion_ejemplo, 0.5)
prob_add_01 = probabilidad_oracion_bigrama_add_k(oracion_ejemplo, 0.1)

# Creamos una tabla comparativa
resultados_suavizado = pd.DataFrame({
    "Modelo": [
        "Bigrama sin suavizado",
        "Bigrama Laplace",
        "Bigrama add-k 0.5",
        "Bigrama add-k 0.1"
    ],
    "Probabilidad": [
        prob_sin_suavizado,
        prob_laplace,
        prob_add_05,
        prob_add_01
    ]
})

resultados_suavizado

,Modelo,Probabilidad
0,Bigrama sin suavizado,1.857352e-05
1,Bigrama Laplace,1.040524e-11
2,Bigrama add-k 0.5,2.855834e-10
3,Bigrama add-k 0.1,9.672920e-08


### Análisis del suavizado en la oración

En esta oración, el modelo bigrama sin suavizado no obtuvo probabilidad cero, lo que indica que los bigramas principales de la oración sí aparecieron en el conjunto de entrenamiento.

Al aplicar suavizado, la probabilidad de la oración disminuye. Esto ocurre porque Laplace y add-k reparten parte de la probabilidad hacia bigramas no vistos.

El suavizado es útil porque evita que una oración completa tenga probabilidad cero cuando aparece una combinación nueva. Aunque en este ejemplo no era necesario para evitar cero, sí muestra cómo el modelo se vuelve menos confiado con las combinaciones vistas.

El ejemplo del bigrama no visto muestra mejor el beneficio práctico del suavizado: sin suavizado la probabilidad fue cero, pero con Laplace y add-k recibió una probabilidad pequeña.